In [2]:
import pandas as pd
pd.read_excel("us_symbol_list.xlsx")

,Symbol
0,BTSG.US
1,CERT.US
2,FSEA.US
3,CURB.US
4,PN.US
...,...
5788,FERA.US
5789,HVII.US
5790,POLE.US
5791,ISRL.US


# 标准化us_symbol_list以及映射到ticker及CIK

In [5]:
import time
import pandas as pd
import requests

# ===== Config =====
USER_AGENT = 'given.family@magnumwm.com'  # 换成你的联系方式（SEC 要求）
EXCEL_PATH = "us_symbol_list.xlsx"

# ===== Fetch SEC mapping with retry =====
def fetch_sec_company_tickers(user_agent, max_retries=5, timeout=30):
    url = "https://www.sec.gov/files/company_tickers.json"
    headers = {"User-Agent": user_agent}
    for attempt in range(max_retries):
        resp = requests.get(url, headers=headers, timeout=timeout)
        if resp.status_code == 200:
            return resp.json()
        time.sleep(1.5 * (2 ** attempt))
    resp.raise_for_status()

sec_raw = fetch_sec_company_tickers(USER_AGENT)

# ===== Build mapping DataFrame =====
sec_df = pd.DataFrame.from_dict(sec_raw, orient="index")
sec_df = sec_df.rename(columns={"cik_str": "cik", "ticker": "ticker", "title": "title"})
sec_df["ticker"] = sec_df["ticker"].astype(str).str.upper().str.strip()
sec_df["cik"] = sec_df["cik"].astype(str).str.zfill(10)

print("SEC mapping columns:", sec_df.columns.tolist())
print("SEC mapping rows:", len(sec_df))
display(sec_df.head(10))

# ===== Load Excel =====
symbols_df = pd.read_excel(EXCEL_PATH)
print("Excel columns:", symbols_df.columns.tolist())
display(symbols_df.head(10))

# ===== Pick symbol column (auto) =====
candidates = ["symbol", "ticker", "Symbol", "Ticker", "SYMBOL", "TICKER"]
symbol_col = next((c for c in candidates if c in symbols_df.columns), symbols_df.columns[0])
print("Using symbol column:", symbol_col)

# 标准化：去空格 + 大写 + 去掉末尾 .US
symbols_df["symbol_norm"] = (
    symbols_df[symbol_col]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(r"\.US$", "", regex=True)
)
# ===== Merge and check =====
merged = symbols_df.merge(
    sec_df[["ticker", "cik", "title"]],
    left_on="symbol_norm",
    right_on="ticker",
    how="left",
)

missing = merged[merged["cik"].isna()]

print("Total symbols:", len(symbols_df))
print("Matched symbols:", merged["cik"].notna().sum())
print("Missing symbols:", len(missing))

display(missing.head(20))
display(merged[merged["cik"].notna()].head(20))


SEC mapping columns: ['cik', 'ticker', 'title']
SEC mapping rows: 10412


,cik,ticker,title
0,0001045810,NVDA,NVIDIA CORP
1,0000320193,AAPL,Apple Inc.
2,0001652044,GOOGL,Alphabet Inc.
3,0000789019,MSFT,MICROSOFT CORP
4,0001018724,AMZN,AMAZON COM INC
5,0001326801,META,"Meta Platforms, Inc."
6,0001318605,TSLA,"Tesla, Inc."
7,0001730168,AVGO,Broadcom Inc.
8,0001067983,BRK-B,BERKSHIRE HATHAWAY INC
9,0000104169,WMT,Walmart Inc.


Excel columns: ['Symbol']


,Symbol
0,BTSG.US
1,CERT.US
2,FSEA.US
3,CURB.US
4,PN.US
5,ANIX.US
6,DWSN.US
7,GDDY.US
8,NMAX.US
9,IFBD.US


Using symbol column: Symbol
Total symbols: 5793
Matched symbols: 5621
Missing symbols: 172


,Symbol,symbol_norm,ticker,cik,title
65,MOG.B.US,MOG.B,NaN,NaN,NaN
125,NISN.US,NISN,NaN,NaN,NaN
155,YGMZ.US,YGMZ,NaN,NaN,NaN
180,JAMF.US,JAMF,NaN,NaN,NaN
211,WSO.B.US,WSO.B,NaN,NaN,NaN
244,FYBR.US,FYBR,NaN,NaN,NaN
250,FATBB.US,FATBB,NaN,NaN,NaN
259,MUE.US,MUE,NaN,NaN,NaN
274,CMPO.US,CMPO,NaN,NaN,NaN
287,BRK.A.US,BRK.A,NaN,NaN,NaN


,Symbol,symbol_norm,ticker,cik,title
0,BTSG.US,BTSG,BTSG,0001865782,"BrightSpring Health Services, Inc."
1,CERT.US,CERT,CERT,0001827090,"Certara, Inc."
2,FSEA.US,FSEA,FSEA,0001943802,"First Seacoast Bancorp, Inc."
3,CURB.US,CURB,CURB,0002027317,Curbline Properties Corp.
4,PN.US,PN,PN,0002001288,Skycorp Solar Group Ltd
5,ANIX.US,ANIX,ANIX,0000715446,Anixa Biosciences Inc
6,DWSN.US,DWSN,DWSN,0000799165,DAWSON GEOPHYSICAL CO
7,GDDY.US,GDDY,GDDY,0001609711,GoDaddy Inc.
8,NMAX.US,NMAX,NMAX,0002026478,Newsmax Inc.
9,IFBD.US,IFBD,IFBD,0001815566,"Infobird Co., Ltd"


In [6]:
# missing 已经是 merged[merged["cik"].isna()]
print("Missing count:", len(missing))
display(missing[[symbol_col, "symbol_norm"]].head(50))


Missing count: 172


,Symbol,symbol_norm
65,MOG.B.US,MOG.B
125,NISN.US,NISN
155,YGMZ.US,YGMZ
180,JAMF.US,JAMF
211,WSO.B.US,WSO.B
244,FYBR.US,FYBR
250,FATBB.US,FATBB
259,MUE.US,MUE
274,CMPO.US,CMPO
287,BRK.A.US,BRK.A


In [ ]:
import time
import requests
import pandas as pd

USER_AGENT = 'given.family@magnumwm.com'  # 换成你的联系方式（SEC 要求）

def get_json(url, max_retries=5, timeout=30):
    headers = {"User-Agent": USER_AGENT}
    for attempt in range(max_retries):
        resp = requests.get(url, headers=headers, timeout=timeout)
        if resp.status_code == 200:
            return resp.json()
        time.sleep(1.5 * (2 ** attempt))
    resp.raise_for_status()

def get_cik_by_ticker(ticker):
    mapping = get_json("https://www.sec.gov/files/company_tickers.json")
    for _, v in mapping.items():
        if v.get("ticker", "").upper() == ticker.upper():
            return str(v.get("cik_str")).zfill(10)
    return None

def fetch_filing_metadata(ticker, forms=None, max_rows=5):
    cik = get_cik_by_ticker(ticker)
    if not cik:
        return pd.DataFrame()

    data = get_json(f"https://data.sec.gov/submissions/CIK{cik}.json")
    recent = data.get("filings", {}).get("recent", {})
    if not recent:
        return pd.DataFrame()

    # recent 是“列式结构”，转成 DataFrame
    df = pd.DataFrame(recent)
    df["cik"] = cik
    df["ticker"] = ticker.upper()

    # 只保留需要的字段
    cols = ["cik", "ticker", "form", "filingDate", "acceptanceDateTime", "accessionNumber"]
    df = df[cols]

    if forms:
        df = df[df["form"].isin(forms)]

    return df.head(max_rows)

# ===== 示例：取几个 ticker 的前几行 =====
tickers = ["AAPL", "MSFT", "NVDA"]
forms = None  # 例如只要 Form 4 和 4/A： ["4", "4/A"]

rows = []
for t in tickers:
    rows.append(fetch_filing_metadata(t, forms=forms, max_rows=5))

result = pd.concat(rows, ignore_index=True)
display(result)


,cik,ticker,form,filingDate,acceptanceDateTime,accessionNumber
0,0000320193,AAPL,4,2026-02-26,2026-02-26T18:34:19.000Z,0001059235-26-000004
1,0000320193,AAPL,4,2026-02-26,2026-02-26T18:33:49.000Z,0001216519-26-000004
2,0000320193,AAPL,4,2026-02-26,2026-02-26T18:33:14.000Z,0001179864-26-000004
3,0000320193,AAPL,4,2026-02-26,2026-02-26T18:32:41.000Z,0001214128-26-000004
4,0000320193,AAPL,4,2026-02-26,2026-02-26T18:32:04.000Z,0001051401-26-000004
5,0000789019,MSFT,4,2026-02-18,2026-02-18T23:14:22.000Z,0000789019-26-000028
6,0000789019,MSFT,4,2026-02-17,2026-02-17T23:06:05.000Z,0000789019-26-000026
7,0000789019,MSFT,4,2026-02-03,2026-02-03T23:07:25.000Z,0000789019-26-000024
8,0000789019,MSFT,4,2026-02-03,2026-02-03T23:06:35.000Z,0000789019-26-000023
9,0000789019,MSFT,4,2026-02-03,2026-02-03T23:06:05.000Z,0000789019-26-000022


In [1]:
import re
import time
import pandas as pd
import requests
from io import StringIO

USER_AGENT = 'given.family@magnumwm.com'   # 按 SEC 要求填写联系方式
BASE_URL = "https://www.sec.gov/submit-filings/forms-index"
OUT_PATH = "sec_forms_index.csv"

def fetch_html(page_num):
    url = BASE_URL if page_num == 0 else f"{BASE_URL}?page={page_num}"
    r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=30)
    r.raise_for_status()
    return r.text

# 先抓第一页找最大页数
html0 = fetch_html(0)
pages = [int(p) for p in re.findall(r"page=(\d+)", html0)]
max_page = max(pages) if pages else 0

tables = []
for page in range(0, max_page + 1):
    html = fetch_html(page)
    dfs = pd.read_html(StringIO(html))
    df = next((t for t in dfs if "Number" in t.columns and "Description" in t.columns), None)
    if df is not None:
        df["page"] = page + 1
        tables.append(df)
    time.sleep(0.3)

forms_df = pd.concat(tables, ignore_index=True)
forms_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print("Saved:", OUT_PATH)
print("Rows:", len(forms_df), "Unique forms:", forms_df["Number"].nunique())
display(forms_df.head(10))


Saved: sec_forms_index.csv
Rows: 165 Unique forms: 148


,Number,Description,Last Updated,SEC Number,Topic(s),page
0,NaN,Examination Brochure: Information about Examin...,March 2023,SEC 2389,NaN,1
1,NaN,Supplemental Information for Persons Requested...,April 2021,SEC2866,Other Forms and Materials,1
2,1,Application for registration or exemption from...,Feb. 1999,SEC1935,Self-Regulatory Organizations,1
3,1-A,Regulation A Offering Statement (PDF),Feb. 2025,SEC486,"Securities Act of 1933, Small Businesses",1
4,1-E,Notification under Regulation E (PDF),Aug. 2001,SEC1807,"Investment Company Act of 1940, Small Business...",1
5,1-K,Annual Reports and Special Financial Reports (...,Feb. 2025,SEC2913,"Securities Act of 1933, Small Businesses",1
6,1-N,Form and amendments for notice of registration...,Dec. 2013,SEC2568,"Securities Exchange Act of 1934, Self-Regulato...",1
7,1-SA,Semiannual Report or Special Financial Report ...,Jan. 2021,SEC2914,"Securities Act of 1933, Small Businesses",1
8,1-U,Current Report Pursuant to Regulation A (PDF),Feb. 2025,SEC2915,"Securities Act of 1933, Small Businesses",1
9,1-Z,Exit Report Under Regulation A (PDF),June 2015,SEC2916,"Securities Act of 1933, Small Businesses",1


In [13]:
import pandas as pd

df = pd.read_csv("sec_forms_index.csv")
forms_list = df.iloc[:, 0].dropna().astype(str).str.strip().unique()
for f in forms_list:
    print(f)


1
1-A
1-E
1-K
1-N
1-SA
1-U
1-Z
10
10-D
10-K
10-M
10-Q
11-K
12b-25
13F
13H
144
15
15F
17-H
18
18-K
19b-4
19b-7
2-E
20-F
24F-2
25
3
4
40-F
5
6-K
7-M
8-A
8-K
8-M
9-M
ABS DD-15E
ABS-15G
ABS-EE
ADV
ADV-E
ADV-H
ADV-NR
ADV-W
ATS
ATS-N
ATS-R
BD
BD-N
BDW
C
CA-1
CB
CFPORTAL
CRS
CUSTODY
D
F-1
F-10
F-3
F-4
F-6
F-7
F-8
F-80
F-N
F-X
ID
MA
MA-I
MA-NR
MA-W
MSD
MSDW
N-14
N-17D-1
N-17f-1
N-17f-2
N-18f-1
N-1A
N-2
N-23c-3
N-27D-1
N-3
N-4
N-5
N-54A
N-54C
N-6
N-6EI-1
N-6F
N-8A
N-8B-2
N-8B-4
N-8F
N-CEN
N-CR
N-CSR
N-MFP
N-PORT
N-PX
N-RN
NRSRO
PF
PILOT
R31
S-1
S-11
S-20
S-3
S-4
S-6
S-8
SBSE
SBSE-A
SBSE-BD
SBSE-C
SBSE-W
SBSEF
SCI
SD
SDR
SE
SF-1
SF-3
SIP
T-1
T-2
T-3
T-4
T-6
TA-1
TA-2
TA-W
TCR
TH
WB-APP
X-17A-19
X-17A-5 Part I
X-17A-5 Part II
X-17A-5 Part IIA
X-17A-5 Part IIC
X-17A-5 Part III
X-17A-5 Schedule I
X-17F-1A
